# Differentiable CBCT (Colab runner)

Run top to bottom on a **GPU** runtime. Data live on Google Drive; this notebook pulls **`main`** from GitHub and writes checkpoints/CSVs under Drive.

In [ ]:
# === 1. TensorFlow & GPU ===
import tensorflow as tf

print('=== TensorFlow & GPU ===')
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus)
if not gpus:
    raise RuntimeError('No GPU found. Use Runtime → Change runtime type → GPU.')

In [ ]:
# === 2. Clone or pull main ===
import os

REPO_URL = 'https://github.com/nkrishnasridhar/differentiable-ct.git'
REPO_DIR = '/content/differentiable-ct'
BRANCH = 'main'

print('=== Git: pull', BRANCH, '===')
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
%cd {REPO_DIR}
print('Working directory:', os.getcwd())

In [ ]:
# === 3. Dependencies ===
print('=== Dependencies ===')
try:
    import skimage  # noqa: F401
    print('scikit-image OK')
except ImportError:
    !pip install -q scikit-image
    print('Installed scikit-image')

In [ ]:
# === 4. Mount Drive & paths ===
from google.colab import drive

print('=== Google Drive ===')
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/cosc440/ct_data/chest_50.pickle'
CHECKPOINT_DIR = '/content/drive/MyDrive/cosc440/checkpoints'
SEED = 0
MODEL = 'nearest'  # or 'trilinear' for Model 2

print('Dataset:', DATASET_PATH)
print('Checkpoints:', CHECKPOINT_DIR)
print('Model:', MODEL)

In [ ]:
# === 5. Unit test ===
print('=== Unit test ===')
!python project_unit_test.py

In [ ]:
# === 6. Smoke training (2 epochs) ===
import os
import sys

sys.path.insert(0, REPO_DIR)
from project_main import main

SMOKE_CSV = os.path.join(REPO_DIR, 'results', 'csv', 'nearest_chest_seed0_smoke.csv')

print('=== Smoke run (nearest, 2 epochs) ===')
main(
    dataset_path=DATASET_PATH,
    epochs=2,
    n_points=32,
    n_rays=256,
    seed=SEED,
    log_csv=SMOKE_CSV,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=False,
    model=MODEL,
)
print('Smoke CSV:', SMOKE_CSV)

In [ ]:
# === 7. Full training (optional) ===
RUN_FULL = False  # set True for 50-epoch baseline on chest

if RUN_FULL:
    print('=== Full run (nearest, 50 epochs) ===')
    main(
        dataset_path=DATASET_PATH,
        epochs=50,
        n_points=192,
        n_rays=2048,
        seed=SEED,
        checkpoint_dir=CHECKPOINT_DIR,
        resume=True,
        model=MODEL,
    )
else:
    print('Skipping full run (RUN_FULL=False).')

In [ ]:
# === 8. Metrics table ===
import os
import pandas as pd

default_csv = os.path.join(REPO_DIR, 'results', 'csv', 'nearest_chest_seed0.csv')
csv_path = SMOKE_CSV if os.path.isfile(SMOKE_CSV) else default_csv

print('=== Metrics CSV ===')
print(csv_path)
df = pd.read_csv(csv_path)
display(df)

In [ ]:
# === 9. Mid-volume slice (from latest checkpoint) ===
import matplotlib.pyplot as plt
import numpy as np
from geometry import TIGREDataset
from project_main import (
    MultiScaleVoxelModel,
    _build_model,
    _checkpoint_run_dir,
    _latest_checkpoint,
    evaluate_volume_metrics,
)

print('=== Mid-volume slice ===')
run_id = f'{MODEL}_chest_seed{SEED}'
checkpoint_path, last_epoch = _latest_checkpoint(_checkpoint_run_dir(CHECKPOINT_DIR, run_id))
if checkpoint_path is None:
    print('No checkpoint found; run smoke or full training first.')
else:
    dataset = TIGREDataset(DATASET_PATH, device='cuda', n_rays=256)
    ground_truth = (dataset.ground_truth.transpose((2, 0, 1)) * 255).astype(np.uint8)
    size = dataset.far - dataset.near
    model = MultiScaleVoxelModel(size, 8, 1, lookup_mode=MODEL)
    _build_model(model)
    model.load_weights(checkpoint_path)
    predicted_volume, ssim, psnr, pred_max, frac_neg = evaluate_volume_metrics(
        model, dataset, ground_truth
    )
    mid = predicted_volume.shape[0] // 2
    print(f'Loaded {checkpoint_path} (epoch {last_epoch})')
    print(f'SSIM={ssim:.4f} PSNR={psnr:.2f} pred_max={pred_max:.4f} frac_negative={frac_neg:.4f}')
    plt.figure(figsize=(5, 5))
    plt.imshow(predicted_volume[mid], cmap='gray')
    plt.title(f'Predicted axial slice z={mid} (epoch {last_epoch})')
    plt.axis('off')
    plt.show()